# Lab 03 — Geometric Transformations

Walking down the hierarchy: **Linear → Affine → Similarity/Rigid → General Affine → Projective**.

Every transformation below is built **by hand with NumPy** (the matrix entries are typed out or
solved from equations). OpenCV is used only to *apply* the matrix to the pixels
(`cv2.warpAffine` / `cv2.warpPerspective`) and, in a few places, as an independent check that the
hand-derived matrix matches the library's.

**Before running:** put your own `image.jpg` next to this notebook. If it is missing, every cell
falls back to a synthetic image generated in the setup cell, so the notebook still runs end to end.

Tasks 9 and 10 need extra imagery (two camera views of a field, a museum wall with an empty frame).
Those are generated synthetically inside their sections — swap in real photos if you have them.


## 0. Setup — imports, image loading, and shared helpers

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

IMAGE_PATH = "image.jpg"          # <-- your image goes here


def make_placeholder(w=480, h=480):
    """Fallback image. Deliberately asymmetric in both axes, so rotation,
    shear and reflection are immediately visible."""
    img = np.full((h, w, 3), 245, np.uint8)
    cv2.rectangle(img, (2, 2), (w - 3, h - 3), (60, 60, 60), 3)
    # A big letter 'F' - no symmetry at all
    cv2.rectangle(img, (120,  90), (160, 390), (180, 90, 20), -1)
    cv2.rectangle(img, (160,  90), (330, 130), (180, 90, 20), -1)
    cv2.rectangle(img, (160, 210), (290, 250), (180, 90, 20), -1)
    cv2.circle(img, (400, 415), 26, (40, 40, 200), -1)          # corner marker
    cv2.putText(img, "LAB 03", (110, 445), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (30, 30, 30), 2)
    return img


def load_image(path=IMAGE_PATH):
    img = cv2.imread(path)
    if img is None:
        print(f"'{path}' not found -> using the built-in synthetic image.")
        img = make_placeholder()
    return img


def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show(items, cols=None, scale=5.0):
    """items: list of (image_bgr, title)"""
    n = len(items)
    cols = cols or n
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(scale * 1.2 * cols, scale * rows))
    for i, (img, title) in enumerate(items, 1):
        plt.subplot(rows, cols, i)
        plt.imshow(to_rgb(img))
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


image = load_image()
H_IMG, W_IMG = image.shape[:2]
print("image shape (h, w):", H_IMG, W_IMG)
show([(image, "Original image")], scale=5)

### Shared matrix toolkit

Everything is kept in **3×3 homogeneous form** so that transformations at different levels of the
hierarchy can be multiplied together. A 2×2 linear map is just the top-left block with
`[0, 0, 1]` pinned to the bottom row.

Order matters: `A @ B` means *apply B first, then A*.

In [ ]:
def M_translate(tx, ty):
    return np.array([[1, 0, tx],
                     [0, 1, ty],
                     [0, 0,  1]], dtype=np.float64)


def M_scale(sx, sy):
    return np.array([[sx, 0, 0],
                     [0, sy, 0],
                     [0,  0, 1]], dtype=np.float64)


def M_rotate(deg):
    t = np.deg2rad(deg)
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]], dtype=np.float64)


def M_shear(kx=0.0, ky=0.0):
    return np.array([[1, kx, 0],
                     [ky, 1, 0],
                     [0,  0, 1]], dtype=np.float64)


def lift(A2x2):
    """Embed a 2x2 linear matrix into 3x3 homogeneous form."""
    M = np.eye(3)
    M[:2, :2] = np.asarray(A2x2, np.float64)
    return M


def about_point(M, cx, cy):
    """Conjugate M so it acts around (cx, cy) instead of the origin."""
    return M_translate(cx, cy) @ M @ M_translate(-cx, -cy)


def apply_to_points(M, pts):
    """Map Nx2 points through a 3x3 matrix, dividing by the homogeneous coordinate."""
    pts = np.asarray(pts, np.float64).reshape(-1, 2)
    hom = np.hstack([pts, np.ones((len(pts), 1))])
    out = (M @ hom.T).T
    return out[:, :2] / out[:, 2:3]


def corners_of(img):
    h, w = img.shape[:2]
    return np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], np.float64)


def warp(img, M, size, border=(255, 255, 255)):
    """Apply a 3x3 matrix. Falls back to warpAffine when the last row is [0,0,1]
    (cheaper, and it is what the affine tasks are meant to use)."""
    M = np.asarray(M, np.float64)
    W, H = int(size[0]), int(size[1])
    common = dict(flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=border)
    if np.allclose(M[2], [0, 0, 1]):
        return cv2.warpAffine(img, M[:2, :], (W, H), **common)
    return cv2.warpPerspective(img, M, (W, H), **common)


def fit_canvas(M, img, pad=0):
    """Translate M so the entire warped image fits inside a tight canvas.
    Returns (M_fitted, (W, H))."""
    p = apply_to_points(M, corners_of(img))
    xmin, ymin = p.min(axis=0)
    xmax, ymax = p.max(axis=0)
    M_fit = M_translate(-xmin + pad, -ymin + pad) @ M
    W = int(np.ceil(xmax - xmin)) + 2 * pad + 1
    H = int(np.ceil(ymax - ymin)) + 2 * pad + 1
    return M_fit, (W, H)


def mask_of(img, M, size):
    """Binary coverage mask of where `img` lands after warping - used for compositing."""
    white = np.full(img.shape[:2] + (3,), 255, np.uint8)
    m = warp(white, M, size, border=(0, 0, 0))
    return cv2.cvtColor(m, cv2.COLOR_BGR2GRAY) > 127


print("helpers ready")

---
## Task 1 — The Tiny Fingerprint (Linear · Scale)

A pure scale is a 2×2 linear map:

$$S=\begin{bmatrix} s_x & 0 \\ 0 & s_y\end{bmatrix}$$

A linear map always fixes the origin, so scaling on its own drags the image towards the top-left
corner. To keep the fingerprint centred, scale **about the image centre** — conjugate `S` with a
translation:

$$M = T(c_x, c_y)\; S\; T(-c_x, -c_y)
    = \begin{bmatrix} s_x & 0 & (1-s_x)c_x \\ 0 & s_y & (1-s_y)c_y \\ 0&0&1 \end{bmatrix}$$

So the correction term is exactly $(1-s)\,c$ on each axis.

**Note on the wording:** "enlarge by 300%" is ambiguous — it can mean *×3* or *+300% ⇒ ×4*.
I read it as ×3 (`scale = 3.0`); change `sx`, `sy` if your instructor meant ×4.

In [ ]:
# --- the raw 2x2 linear matrix, typed out by hand ---
sx, sy = 3.0, 3.0
S2 = np.array([[sx, 0.0],
               [0.0, sy]])
print("2x2 scaling matrix S =\n", S2)

cx, cy = (W_IMG - 1) / 2, (H_IMG - 1) / 2       # centre of the current "screen"

# Option A - same window size, magnified about the centre (fingerprint stays centred, edges crop)
M_zoom = about_point(lift(S2), cx, cy)
print("\ncentred scale matrix (3x3):\n", M_zoom)
print("hand-checked offsets: (1-sx)*cx =", (1 - sx) * cx, " (1-sy)*cy =", (1 - sy) * cy)
zoom_same = warp(image, M_zoom, (W_IMG, H_IMG))

# Option B - let the canvas grow with the image, nothing is lost
M_grow, size_grow = fit_canvas(lift(S2), image)
zoom_grow = warp(image, M_grow, size_grow)

print("\ncanvas A:", (W_IMG, H_IMG), " canvas B:", size_grow)
show([(image, "Original"),
      (zoom_same, f"A: {sx:g}x about centre (same canvas)"),
      (zoom_grow, f"B: {sx:g}x with canvas grown")])

---
## Task 2 — The Dizzy Satellite (Linear · Rotation)

$$R(\theta)=\begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}$$

Image coordinates have **y pointing down**, so with this matrix a positive $\theta$ turns the
picture *clockwise* on screen. The satellite's photo is tilted by +45°, so the correction is −45°.

To stop the corners being chopped off, the canvas has to grow to the bounding box of the rotated
rectangle:

$$W' = |w\cos\theta| + |h\sin\theta|, \qquad H' = |w\sin\theta| + |h\cos\theta|$$

and the rotation is then centred on the *new* canvas:
$M = T(W'/2,\,H'/2)\; R\; T(-w/2,\,-h/2)$.

In [ ]:
def rotate_no_crop(img, deg):
    """Rotate about the centre, growing the canvas so no corner is lost.
    Matrix built by hand from sin/cos."""
    h, w = img.shape[:2]
    t = np.deg2rad(deg)
    c, s = np.cos(t), np.sin(t)

    R2 = np.array([[c, -s],
                   [s,  c]])                       # the 2x2 rotation matrix

    new_w = int(np.ceil(abs(w * c) + abs(h * s)))  # manual bounding-box size
    new_h = int(np.ceil(abs(w * s) + abs(h * c)))

    M = M_translate(new_w / 2, new_h / 2) @ lift(R2) @ M_translate(-w / 2, -h / 2)
    return warp(img, M, (new_w, new_h)), M, (new_w, new_h)


# 1) simulate the damage: the satellite photographed the city at +45 degrees
tilted, M_tilt, _ = rotate_no_crop(image, 45)

# 2) spin it back to true north
fixed, M_fix, size_fix = rotate_no_crop(tilted, -45)

print("rotation matrix used for the fix:\n", M_fix)
print("canvas grows to keep every corner:", image.shape[1::-1], "->", tilted.shape[1::-1],
      "->", size_fix)

# independent check against OpenCV's own builder (same centre, no canvas growth)
M_cv = cv2.getRotationMatrix2D(((W_IMG - 1) / 2, (H_IMG - 1) / 2), 45, 1.0)
M_mine = about_point(M_rotate(-45), (W_IMG - 1) / 2, (H_IMG - 1) / 2)[:2, :]
print("\nmatches cv2.getRotationMatrix2D(angle=45):", np.allclose(M_cv, M_mine, atol=1e-9))
print("(cv2's positive angle is counter-clockwise, which is my -45)")

show([(image, "Original"), (tilted, "Damaged: +45 deg"), (fixed, "Corrected: -45 deg, no crop")])

---
## Task 3 — The Fast Train Barcode (Linear · Shear)

An x-shear pushes pixels sideways in proportion to their height:

$$H_x(k)=\begin{bmatrix}1 & k\\ 0 & 1\end{bmatrix}, \qquad x' = x + k\,y,\; y'=y$$

Undoing a slant of $+k$ is just $H_x(-k)$ — the inverse of a shear is a shear with the opposite
factor. The canvas also needs $|k|\cdot h$ extra width, otherwise the leaning content is clipped.

In [ ]:
k = 0.4     # slant introduced by the train's motion

Sh = np.array([[1.0, k],
               [0.0, 1.0]])
print("2x2 shear matrix =\n", Sh)
print("manual new width  = w + |k|*h =", W_IMG, "+", abs(k) * H_IMG, "=", W_IMG + abs(k) * H_IMG)

# 1) simulate the slanted scan
M_slant, size_slant = fit_canvas(lift(Sh), image)
slanted = warp(image, M_slant, size_slant)

# 2) push it back into a rectangle with the inverse shear
Sh_inv = np.array([[1.0, -k],
                   [0.0, 1.0]])
print("\ninverse shear =\n", Sh_inv, "\nSh @ Sh_inv = identity:", np.allclose(Sh @ Sh_inv, np.eye(2)))

# the slanted scan was rendered on a shifted canvas, so the full undo is
# inv(M_slant) = Sh_inv followed by that same shift removed - its linear part is exactly Sh_inv
M_unshear = np.linalg.inv(M_slant)
print("linear part of the undo matrix is the inverse shear:",
      np.allclose(M_unshear[:2, :2], Sh_inv))
unslanted = warp(slanted, M_unshear, (W_IMG, H_IMG))
print("recovered size", (W_IMG, H_IMG), "= original", (W_IMG, H_IMG))

show([(image, "Original barcode"),
      (slanted, f"Scanned while moving (k={k})"),
      (unslanted, "De-sheared")])

---
## Task 4 — The Hidden Treasure Map (Affine · Translation)

Translation cannot be written as a 2×2 matrix — that is the whole reason for moving up to
homogeneous 3×3 coordinates:

$$T=\begin{bmatrix}1&0&t_x\\0&1&t_y\\0&0&1\end{bmatrix}$$

The X is off-screen 150 px to the **left** and 80 px **up**, so it is brought back with
$t_x=+150$, $t_y=+80$ (y grows downwards, so "up" is a negative y and the fix is positive).

In [ ]:
# put an "X marks the spot" near the top-left so we can watch it disappear and return
treasure = image.copy()
X_POS = (95, 70)
cv2.line(treasure, (X_POS[0] - 28, X_POS[1] - 28), (X_POS[0] + 28, X_POS[1] + 28), (0, 0, 255), 6)
cv2.line(treasure, (X_POS[0] - 28, X_POS[1] + 28), (X_POS[0] + 28, X_POS[1] - 28), (0, 0, 255), 6)

# 1) the misalignment: the map sits 150 px left and 80 px up, so the X is outside the frame
T_bad = M_translate(-150, -80)
misaligned = warp(treasure, T_bad, (W_IMG, H_IMG))

# 2) the fix
tx, ty = 150, 80
T_fix = M_translate(tx, ty)
print("translation matrix:\n", T_fix)
recovered = warp(misaligned, T_fix, (W_IMG, H_IMG))

print("X was at", X_POS,
      "-> after misalignment:", apply_to_points(T_bad, [X_POS])[0],
      "-> after fix:", apply_to_points(T_fix @ T_bad, [X_POS])[0])

show([(treasure, "Map as it should be"),
      (misaligned, "Misaligned: X is off-screen"),
      (recovered, f"Translated by ({tx}, {ty})")])

---
## Task 5 — The Robot's Assembly Line (Rigid Transformation)

A rigid (Euclidean) transformation is rotation **+** translation only:

$$M=\begin{bmatrix}\cos\theta & -\sin\theta & t_x\\ \sin\theta & \cos\theta & t_y\\ 0&0&1\end{bmatrix}$$

4 degrees of freedom, and it preserves *everything* about the shape: lengths, angles, areas,
handedness. Diagnostics for "is this really rigid": the upper-left 2×2 block is orthogonal
($R^TR=I$) and $\det = +1$.

The two operations are baked into **one** 3×3 matrix and applied in a single warp, so the chip is
never resampled twice.

In [ ]:
theta   = 30.0            # chip is rotated sideways
tx, ty  = 100.0, 50.0     # ... and slightly off-centre
cxp, cyp = (W_IMG - 1) / 2, (H_IMG - 1) / 2

# built by hand, one matrix, one warp
t = np.deg2rad(theta)
c, s = np.cos(t), np.sin(t)
R_rigid = np.array([[c, -s],
                    [s,  c]])

M_rigid = M_translate(tx, ty) @ about_point(lift(R_rigid), cxp, cyp)
print("single 3x3 rigid matrix:\n", M_rigid)

A = M_rigid[:2, :2]
print("\northogonal (R^T R = I):", np.allclose(A.T @ A, np.eye(2)))
print("determinant           :", round(float(np.linalg.det(A)), 6), "(must be +1)")

# distances between the four corners must be untouched
p0 = corners_of(image)
p1 = apply_to_points(M_rigid, p0)
d0 = np.linalg.norm(p0 - np.roll(p0, 1, axis=0), axis=1)
d1 = np.linalg.norm(p1 - np.roll(p1, 1, axis=0), axis=1)
print("edge lengths preserved:", np.allclose(d0, d1), " ->", d0.round(2), d1.round(2))

M_fit, size = fit_canvas(M_rigid, image)
rigid_img = warp(image, M_fit, size)
show([(image, "Chip as seen"), (rigid_img, f"Rigid: rotate {theta:g} deg + translate ({tx:g},{ty:g})")])

---
## Task 6 — The Architect's Blueprint (Similarity Transformation)

Similarity = **scale · rotation · translation**, 4 DOF:

$$M=\begin{bmatrix}s\cos\theta & -s\sin\theta & t_x\\ s\sin\theta & s\cos\theta & t_y\\ 0&0&1\end{bmatrix}$$

Angles survive, lengths do not — every distance is multiplied by the same $s$, which is why the
blueprint stays *shaped* like the blueprint. Diagnostics: $A^TA = s^2 I$ and $\det A = s^2$.

Composition order below is `T @ R @ S` — scale first, then rotate, then move.

In [ ]:
s_fac  = 1.6
theta  = -20.0
tx, ty = 60.0, 30.0

M_sim = M_translate(tx, ty) @ M_rotate(theta) @ M_scale(s_fac, s_fac)
print("single 3x3 similarity matrix:\n", M_sim)

A = M_sim[:2, :2]
print("\nA^T A = s^2 I :", np.allclose(A.T @ A, (s_fac ** 2) * np.eye(2)))
print("det(A)        :", round(float(np.linalg.det(A)), 6), " expected s^2 =", s_fac ** 2)

# angle preservation: the two edges at a corner are perpendicular before and after
p = apply_to_points(M_sim, corners_of(image))
e1, e2 = p[1] - p[0], p[3] - p[0]
print("corner angle stays 90 deg:", abs(float(np.dot(e1, e2))) < 1e-6)
print("all lengths scaled by s   :",
      np.allclose(np.linalg.norm(p - np.roll(p, 1, axis=0), axis=1),
                  s_fac * np.linalg.norm(corners_of(image) - np.roll(corners_of(image), 1, axis=0), axis=1)))

M_fit, size = fit_canvas(M_sim, image)
sim_img = warp(image, M_fit, size)
show([(image, "Imported blueprint"),
      (sim_img, f"Similarity: scale {s_fac}, rotate {theta:g} deg, move ({tx:g},{ty:g})")])

---
## Task 7 — The Glitched Image (General Affine)

A general affine has **6 unknowns** and no constraints on the 2×2 block:

$$\begin{bmatrix}x'\\y'\end{bmatrix}=
\begin{bmatrix}a&b\\c&d\end{bmatrix}\begin{bmatrix}x\\y\end{bmatrix}+\begin{bmatrix}e\\f\end{bmatrix}$$

Each landmark correspondence gives two equations, so **3 non-collinear points** pin it down exactly.
Stacking them:

$$\begin{bmatrix}
x_1&y_1&1&0&0&0\\ 0&0&0&x_1&y_1&1\\ &&\vdots&&&
\end{bmatrix}
\begin{bmatrix}a\\b\\e\\c\\d\\f\end{bmatrix}=
\begin{bmatrix}x'_1\\y'_1\\ \vdots\end{bmatrix}$$

a 6×6 system solved with `np.linalg.solve`. Here the landmarks are read *from the glitched image*
and mapped *to* their known original positions, so the solution is directly the repair matrix.

(If the 3 points are collinear the matrix is singular and `np.linalg.solve` raises
`LinAlgError` — that is the degenerate case to avoid when picking landmarks.)

In [ ]:
def affine_from_3_points(src, dst):
    """Solve the 6 unknowns of an affine map taking src -> dst. Pure linear algebra."""
    src = np.asarray(src, np.float64)
    dst = np.asarray(dst, np.float64)
    A, b = [], []
    for (x, y), (u, v) in zip(src, dst):
        A.append([x, y, 1, 0, 0, 0]); b.append(u)
        A.append([0, 0, 0, x, y, 1]); b.append(v)
    sol = np.linalg.solve(np.array(A), np.array(b))
    return np.array([[sol[0], sol[1], sol[2]],
                     [sol[3], sol[4], sol[5]],
                     [0.0,    0.0,    1.0   ]])


# --- the corruption: squish + slant + rotate + move, all at once ---
A_glitch = np.array([[ 1.10, 0.45,  30.0],
                     [-0.25, 0.90,  60.0],
                     [ 0.00, 0.00,   1.0]])
A_glitch_fit, size_g = fit_canvas(A_glitch, image)
glitched = warp(image, A_glitch_fit, size_g)

# --- 3 landmarks: where they belong, and where the corruption put them ---
landmarks_true = np.array([[60.0, 70.0],
                           [W_IMG - 80.0, 50.0],
                           [100.0, H_IMG - 60.0]])
landmarks_seen = apply_to_points(A_glitch_fit, landmarks_true)
print("landmarks in the glitched photo:\n", landmarks_seen)

# --- solve for the repair matrix: glitched coords -> original coords ---
A_fix = affine_from_3_points(landmarks_seen, landmarks_true)
print("\nrecovered affine (6 unknowns):\n", A_fix)
print("\nmatches the true inverse:", np.allclose(A_fix, np.linalg.inv(A_glitch_fit), atol=1e-6))
print("cross-check vs cv2.getAffineTransform:",
      np.allclose(A_fix[:2], cv2.getAffineTransform(landmarks_seen.astype(np.float32),
                                                    landmarks_true.astype(np.float32)), atol=1e-4))

repaired = warp(glitched, A_fix, (W_IMG, H_IMG))
show([(image, "Intended photo"), (glitched, "Corrupted transmission"), (repaired, "Repaired via 3 landmarks")])

---
## Task 8 — The Sidewalk Illusion (Projective · Perspective)

Top of the hierarchy: **8 DOF**, straight lines stay straight but parallel lines are free to meet.

$$\begin{bmatrix}\tilde x'\\ \tilde y'\\ w'\end{bmatrix}=
\begin{bmatrix}h_{11}&h_{12}&h_{13}\\ h_{21}&h_{22}&h_{23}\\ h_{31}&h_{32}&h_{33}\end{bmatrix}
\begin{bmatrix}x\\y\\1\end{bmatrix},\qquad
x'=\frac{\tilde x'}{w'},\; y'=\frac{\tilde y'}{w'}$$

The matrix is only defined up to scale, so fixing $h_{33}=1$ leaves 8 unknowns and
**4 point pairs** (8 equations) determine it. Clearing the denominators gives, per correspondence:

$$x h_{11}+y h_{12}+h_{13}-u x h_{31}-u y h_{32}=u,\qquad
x h_{21}+y h_{22}+h_{23}-v x h_{31}-v y h_{32}=v$$

**Caveat on this shortcut:** setting $h_{33}=1$ fails for the rare homographies whose true
$h_{33}=0$, and it cannot use more than 4 points. The general recipe is the SVD null-space of the
$2N\times9$ matrix (that is what `cv2.findHomography` does). For these labs the 8×8 solve is fine,
and I check it against OpenCV below.

In [ ]:
def homography_from_4_points(src, dst):
    """Solve the 8 unknowns of a projective matrix (h33 fixed to 1) from 4 correspondences."""
    src = np.asarray(src, np.float64).reshape(-1, 2)
    dst = np.asarray(dst, np.float64).reshape(-1, 2)
    assert len(src) == len(dst) == 4, "exactly 4 point pairs"
    A, b = [], []
    for (x, y), (u, v) in zip(src, dst):
        A.append([x, y, 1, 0, 0, 0, -u * x, -u * y]); b.append(u)
        A.append([0, 0, 0, x, y, 1, -v * x, -v * y]); b.append(v)
    h = np.linalg.solve(np.array(A), np.array(b))
    return np.append(h, 1.0).reshape(3, 3)


# --- simulate standing in front of 3D chalk art: far edge shrinks, near edge stays wide ---
PH_W, PH_H = 620, 520
art_quad_photo = np.float32([[180,  40],      # far-left   (small, up the sidewalk)
                             [440,  40],      # far-right
                             [575, 470],      # near-right (wide, at your feet)
                             [ 45, 470]])     # near-left
H_photo = homography_from_4_points(corners_of(image), art_quad_photo)
photo = warp(image, H_photo, (PH_W, PH_H))

# --- the 4 corners that *should* be a perfect square, mapped to one ---
L = 420
square = np.float32([[0, 0], [L - 1, 0], [L - 1, L - 1], [0, L - 1]])
H_bird = homography_from_4_points(art_quad_photo, square)

print("hand-solved homography:\n", H_bird)
H_cv = cv2.getPerspectiveTransform(art_quad_photo, square)
print("\nmatches cv2.getPerspectiveTransform:", np.allclose(H_bird, H_cv / H_cv[2, 2], atol=1e-6))
print("bottom row is not [0,0,1] -> genuinely projective:", H_bird[2].round(6))

birdseye = warp(photo, H_bird, (L, L))

marked = photo.copy()
cv2.polylines(marked, [art_quad_photo.astype(np.int32)], True, (0, 0, 255), 3)
for p in art_quad_photo.astype(int):
    cv2.circle(marked, tuple(p), 7, (0, 0, 255), -1)

show([(photo, "Photo of the chalk art"),
      (marked, "4 corners that form a square"),
      (birdseye, "Warped to bird's-eye view")])

---
## Task 9 — The Stadium Panorama (Projective · Homography)

Two cameras looking at the same **plane** (the pitch) are related by a homography, so 4 matching
points in the overlap are enough to project camera B into camera A's frame.

Both views are synthesised here from one wide pitch image through two known camera homographies,
$H_A$ and $H_B$. That means the ground truth is available: the correct stitching matrix is
$H_A H_B^{-1}$, and the matrix solved from 4 clicked points should equal it. With real photographs
you would get the correspondences from a feature matcher (SIFT/ORB + `cv2.findHomography` with
RANSAC) instead of by hand.

Blending is deliberately naive — camera A is simply painted over camera B, so the seam is visible.
Exposure blending is a different topic.

In [ ]:
def make_pitch(W=1200, H=600):
    """A synthetic football pitch with enough landmarks to match on."""
    f = np.full((H, W, 3), (60, 130, 55), np.uint8)
    white = (250, 250, 250)
    cv2.rectangle(f, (40, 40), (W - 40, H - 40), white, 4)
    cv2.line(f, (W // 2, 40), (W // 2, H - 40), white, 4)
    cv2.circle(f, (W // 2, H // 2), 90, white, 4)
    cv2.rectangle(f, (40, H // 2 - 150), (210, H // 2 + 150), white, 4)
    cv2.rectangle(f, (W - 210, H // 2 - 150), (W - 40, H // 2 + 150), white, 4)
    for i, (x, y) in enumerate([(300, 150), (300, 450), (620, 130), (620, 470),
                                (900, 150), (900, 450)], start=1):
        cv2.circle(f, (x, y), 13, (0, 0, 210), -1)
        cv2.putText(f, str(i), (x + 20, y + 9), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 210), 2)
    return f


pitch = make_pitch()

# camera A: left 780 px of the pitch, mild perspective
quadA = np.float32([[0, 0], [780, 0], [780, 600], [0, 600]])
viewA = np.float32([[12, 22], [770, 0], [780, 588], [0, 600]])
H_A = homography_from_4_points(quadA, viewA)
imgA = warp(pitch, H_A, (790, 610), border=(0, 0, 0))

# camera B: right side from x=520 (so 520..780 overlaps A), different perspective
quadB = np.float32([[520, 0], [1200, 0], [1200, 600], [520, 600]])
viewB = np.float32([[0, 35], [660, 0], [690, 585], [25, 620]])
H_B = homography_from_4_points(quadB, viewB)
imgB = warp(pitch, H_B, (700, 630), border=(0, 0, 0))

# --- 4 matching points, taken in the overlapping strip (pitch x in [520, 780]) ---
overlap_pts = np.float32([[545,  95], [760,  95], [765, 505], [560, 505]])
ptsA = apply_to_points(H_A, overlap_pts)     # where they appear in camera A
ptsB = apply_to_points(H_B, overlap_pts)     # where they appear in camera B
print("matching points\n  in A:\n", ptsA, "\n  in B:\n", ptsB)

# --- homography that projects camera B into camera A's space ---
H_BA = homography_from_4_points(ptsB, ptsA)
H_true = H_A @ np.linalg.inv(H_B)
print("\nH_BA:\n", H_BA)
print("matches ground truth H_A @ inv(H_B):", np.allclose(H_BA, H_true / H_true[2, 2], atol=1e-4))

# --- common canvas that holds both views ---
allc = np.vstack([corners_of(imgA), apply_to_points(H_BA, corners_of(imgB))])
xmin, ymin = allc.min(0); xmax, ymax = allc.max(0)
Off = M_translate(-xmin, -ymin)
PW, PH = int(np.ceil(xmax - xmin)) + 1, int(np.ceil(ymax - ymin)) + 1
print("panorama canvas:", (PW, PH))

pano   = warp(imgB, Off @ H_BA, (PW, PH), border=(0, 0, 0))
warpedA = warp(imgA, Off, (PW, PH), border=(0, 0, 0))
maskA  = mask_of(imgA, Off, (PW, PH))
pano[maskA] = warpedA[maskA]        # A painted on top of B

show([(imgA, "Camera A"), (imgB, "Camera B")], scale=4.5)
show([(pano, "Stitched panorama (B projected into A's frame)")], scale=7)

---
## Task 10 — The Master Forger (Full Hierarchy)

One chain, walked all the way down:

$$H_{\text{total}} \;=\; \underbrace{P}_{\text{projective}}\;
\underbrace{T\,R}_{\text{rigid}}\;\underbrace{S}_{\text{linear scale}}$$

Read right to left: shrink the painting, move/rotate it up next to the frame, then warp its four
(already moved) corners onto the frame's four corners. The projective stage is solved *from the
placed corners*, so all three stages genuinely contribute to $H_{\text{total}}$ — dropping the scale
or the rigid step changes $P$, and the product still lands on the frame either way.

The painting is finally composited through a coverage mask so the wall around the frame is
untouched.

In [ ]:
def make_museum_wall(W=900, H=650):
    """Gallery wall with one empty, heavily angled frame. Returns (wall, inner_quad)."""
    wall = np.zeros((H, W, 3), np.uint8)
    wall[:] = np.linspace(205, 170, H).reshape(H, 1, 1)          # soft vertical gradient
    cv2.line(wall, (0, H - 70), (W, H - 70), (140, 140, 140), 3)  # floor line
    cv2.putText(wall, "GALLERY 3", (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (120, 120, 120), 2)

    inner = np.float32([[300, 150], [720,  95], [742, 470], [322, 545]])   # angled opening
    c = inner.mean(axis=0)
    outer = (c + (inner - c) * 1.10).astype(np.float32)
    cv2.fillPoly(wall, [outer.astype(np.int32)], (35, 75, 125))   # frame moulding (BGR brown)
    cv2.fillPoly(wall, [inner.astype(np.int32)], (55, 55, 55))    # empty backing board
    cv2.polylines(wall, [inner.astype(np.int32)], True, (20, 20, 20), 2)
    return wall, inner


wall, frame_quad = make_museum_wall()
WALL_W, WALL_H = wall.shape[1], wall.shape[0]
painting = image


def paste(canvas, img, M):
    """Composite img onto a copy of canvas using the warp's coverage mask."""
    out = canvas.copy()
    size = (canvas.shape[1], canvas.shape[0])
    w = warp(img, M, size, border=(0, 0, 0))
    m = mask_of(img, M, size)
    out[m] = w[m]
    return out


# stage 1 - LINEAR: shrink the painting
s = 0.34
S = M_scale(s, s)

# stage 2 - RIGID: rotate a touch and move it beside the frame (shape untouched)
Rg = M_translate(70.0, 210.0) @ about_point(M_rotate(9.0), s * W_IMG / 2, s * H_IMG / 2)

# stage 3 - PROJECTIVE: send the placed corners onto the frame's corners
placed_corners = apply_to_points(Rg @ S, corners_of(painting))
P = homography_from_4_points(placed_corners, frame_quad)

H_total = P @ Rg @ S
print("scale S:\n", S, "\n\nrigid T@R:\n", Rg, "\n\nprojective P:\n", P)
print("\nH_total = P @ Rg @ S:\n", H_total)

landed = apply_to_points(H_total, corners_of(painting))
print("\npainting corners land on the frame:", np.allclose(landed, frame_quad, atol=1e-6))
print("frame corners:\n", frame_quad, "\nlanded:\n", landed)

stage1 = paste(wall, painting, S)                 # scaled only
stage2 = paste(wall, painting, Rg @ S)            # scaled + rigid
final  = paste(wall, painting, H_total)           # + projective

show([(wall, "Empty angled frame"), (painting, "Flat painting")], scale=4.5)
show([(stage1, "1. Linear scale"),
      (stage2, "2. + Rigid move"),
      (final,  "3. + Projective warp into the frame")], scale=4.5)

---
## Recap — the hierarchy in one table

| Level | Matrix form | DOF | Preserves | Task |
|---|---|---|---|---|
| Linear (scale / rotation / shear) | 2×2, origin fixed | 4 | origin, straight lines, parallelism | 1, 2, 3 |
| Translation (affine) | 3×3 with `[0 0 1]` bottom row | 2 | everything but position | 4 |
| Rigid / Euclidean | $[R \mid t]$, $R^TR=I$, $\det=+1$ | 3 | lengths, angles, areas | 5 |
| Similarity | $[sR \mid t]$ | 4 | angles, ratios of lengths | 6 |
| Affine (general) | $[A \mid t]$, $A$ arbitrary | 6 | parallelism, ratios along a line | 7 |
| Projective (homography) | full 3×3 up to scale | 8 | straight lines, cross-ratio | 8, 9, 10 |

Each level is a strict subset of the one below it, and each extra pair of DOF costs one more point
correspondence to solve: 2 points for similarity, 3 for affine, 4 for a homography.

**Minimum correspondences used above:** 3 points → 6×6 solve (Task 7); 4 points → 8×8 solve
(Tasks 8–10). With more points than the minimum, switch to a least-squares / SVD solution
(`np.linalg.lstsq`, `cv2.estimateAffine2D`, `cv2.findHomography` with RANSAC) — those tolerate the
noise that real feature matches always carry.
